# VulnReach DAPT Pipeline — Kaggle (2×T4)

**Hardware:** 2× NVIDIA T4 16 GB (fp16, no bf16)  
**GPU quota:** 30 hrs/week free  

### Before running
1. *Settings → Accelerator → GPU T4 × 2*
2. *Settings → Internet → On*
3. Push your repo to GitHub, set `REPO_URL` in the config cell below
4. **Recommended:** run `python scripts/prepare_kaggle.py` locally first,
   upload the resulting `dapt_kaggle_data.zip` as a Kaggle dataset named **dapt-corpus**,
   add it to this notebook via *Add Data*, then set `DATA_FROM_DATASET = True`.
   This skips ~2 hrs of data prep and lets you start training immediately.

In [ ]:
# ── Config — edit these two lines ────────────────────────────────────────────
REPO_URL         = "https://github.com/YOUR_USERNAME/dapt-code.git"
DATA_FROM_DATASET = True   # True  → data/ already uploaded as Kaggle dataset
                            # False → download corpus live (wastes ~2 hrs GPU time)

# Kaggle dataset name (only used when DATA_FROM_DATASET = True)
# Must match the slug of the dataset you uploaded, e.g. "dapt-corpus"
KAGGLE_DATASET_SLUG = "dapt-corpus"

# Training knobs — T4 fp16, reduced DAPT steps to fit in one session
DAPT_STEPS = 6_000   # full spec = 10 000; resume in a 2nd session if needed
DAPT_BATCH = 2       # 2 per GPU × 2 GPUs × grad_accum 8 = effective 32
SFT_STEPS  = 2_000
SFT_BATCH  = 2
GRAD_ACCUM = 8

# Paths
WORK_DIR  = "/kaggle/working/dapt-code"
INPUT_DIR = f"/kaggle/input/{KAGGLE_DATASET_SLUG}"

## 0 — Environment check

In [ ]:
import torch

n_gpus = torch.cuda.device_count()
assert n_gpus >= 1, "No GPU — enable GPU in Settings"

for i in range(n_gpus):
    p = torch.cuda.get_device_properties(i)
    bf16 = "bf16 ✓" if torch.cuda.is_bf16_supported() else "fp16 only"
    print(f"GPU {i}: {p.name}  {round(p.total_memory/1e9,1)} GB  CC {p.major}.{p.minor}  {bf16}")

USE_BF16 = torch.cuda.is_bf16_supported()
PREC_FLAG = [] if USE_BF16 else ["--fp16"]
print(f"\nPrecision: {'bf16' if USE_BF16 else 'fp16 (T4 mode)'}  |  {n_gpus} GPU(s)")

## 1 — Install dependencies

In [ ]:
# Kaggle pre-installs torch; install the rest quietly
!pip install -q \
    "transformers>=4.40.0" \
    "datasets>=2.18.0" \
    "trl>=0.8.6" \
    "peft>=0.10.0" \
    "accelerate>=0.28.0" \
    "bitsandbytes>=0.43.0" \
    sentencepiece protobuf gitpython tqdm

print("Dependencies installed.")

## 2 — Clone repo

In [ ]:
import os, subprocess

if not os.path.exists(WORK_DIR):
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)

os.chdir(WORK_DIR)
print(f"CWD: {os.getcwd()}")

## 3 — Data setup

**If `DATA_FROM_DATASET = True`** (recommended): symlinks the pre-built data from your
uploaded Kaggle dataset. The uploaded zip was created by `scripts/prepare_kaggle.py`
on your local machine and contains `data/dapt_corpus/`, `data/dapt_tokenized/`,
and `data/sft_dataset/` — all three phases already done.

**If `DATA_FROM_DATASET = False`**: downloads the corpus live and runs all three
prep phases (~2 hrs before training starts).

In [ ]:
import os

os.makedirs("data", exist_ok=True)

if DATA_FROM_DATASET:
    print(f"Mounting pre-built data from {INPUT_DIR} ...")

    # The zip was created from repo root so paths inside are data/dapt_corpus/ etc.
    # Kaggle extracts the zip at INPUT_DIR, so full path is INPUT_DIR/data/<subdir>
    subdirs = {
        "dapt_corpus"   : "data/dapt_corpus",
        "dapt_tokenized": "data/dapt_tokenized",
        "sft_dataset"   : "data/sft_dataset",
    }

    for name, local_path in subdirs.items():
        src = os.path.join(INPUT_DIR, "data", name)
        if not os.path.exists(src):
            # Some uploads flatten the structure — try without "data/" prefix
            src = os.path.join(INPUT_DIR, name)

        if os.path.exists(src):
            if not os.path.exists(local_path):
                os.symlink(src, local_path)
            print(f"  ✓ {local_path} → {src}")
        else:
            print(f"  ✗ {name} not found in dataset — will rebuild")

    # Verify what we have
    missing = [d for d in subdirs.values() if not os.path.exists(d)]
    if missing:
        print(f"\nMissing: {missing} — running prep phases to fill gaps")
    else:
        print("\nAll data directories present. Skipping prep phases.")

else:
    print("DATA_FROM_DATASET=False — will run corpus ingest + tokenize + SFT build.")
    missing = ["data/dapt_corpus", "data/dapt_tokenized", "data/sft_dataset"]

In [ ]:
# Run only the phases whose output is missing
import subprocess

def _run(script):
    r = subprocess.run(["python", script])
    assert r.returncode == 0, f"{script} failed"

if not os.path.exists("data/dapt_corpus"):
    print("=== Phase 1: corpus ingest ===")
    _run("scripts/dapt/ingest_corpus.py")

if not os.path.exists("data/dapt_tokenized"):
    print("=== Phase 1b: merge + tokenize ===")
    _run("scripts/dapt/merge_and_tokenize.py")

if not os.path.exists("data/sft_dataset"):
    print("=== Phase 3: SFT dataset ===")
    _run("scripts/sft/build_sft_dataset.py")

print("\nData ready:")
for d in ["data/dapt_corpus", "data/dapt_tokenized", "data/sft_dataset"]:
    size = sum(f.stat().st_size for f in __import__('pathlib').Path(d).rglob('*') if f.is_file()) / 1e6 if os.path.exists(d) else 0
    print(f"  {d:<30} {size:>8.0f} MB")

## Phase 2 — DAPT training

Runs on **2×T4 via `torchrun`**.  
Precision is set automatically (fp16 on T4, bf16 on A100).  

> **Resume:** if the session times out, just re-run this cell —
> the Trainer detects the latest checkpoint in `models/dapt_checkpoints/` automatically.

In [ ]:
import subprocess, torch

n_gpus = torch.cuda.device_count()
print(f"DAPT: {DAPT_STEPS} steps on {n_gpus} GPU(s) ...")

cmd = [
    "torchrun", f"--nproc_per_node={n_gpus}",
    "scripts/dapt/train_dapt.py",
    f"--max-steps={DAPT_STEPS}",
    f"--batch-size={DAPT_BATCH}",
    f"--grad-accum={GRAD_ACCUM}",
] + PREC_FLAG

r = subprocess.run(cmd)
assert r.returncode == 0, "DAPT failed — check output above"

## Phase 4 — SFT training

In [ ]:
n_gpus = torch.cuda.device_count()
print(f"SFT: {SFT_STEPS} steps on {n_gpus} GPU(s) ...")

cmd = [
    "torchrun", f"--nproc_per_node={n_gpus}",
    "scripts/sft/train_sft.py",
    f"--max-steps={SFT_STEPS}",
    f"--batch-size={SFT_BATCH}",
    f"--grad-accum={GRAD_ACCUM}",
] + PREC_FLAG

r = subprocess.run(cmd)
assert r.returncode == 0, "SFT failed — check output above"

## Phase 5 — Quantize → GGUF

In [ ]:
if os.path.exists("scripts/export/quantize.py"):
    r = subprocess.run(["python", "scripts/export/quantize.py"])
    assert r.returncode == 0, "Quantize failed"
else:
    print("quantize.py not yet written — skipping.")

## Save artefacts for download

Copies adapters (and GGUF if available) to `/kaggle/working/artefacts/`.
After the session ends they appear in the **Output** tab for download.

In [ ]:
import shutil, os

OUT = "/kaggle/working/artefacts"
os.makedirs(OUT, exist_ok=True)

for src, name in [("models/dapt_adapter", "dapt_adapter"), ("models/sft_final", "sft_final")]:
    dst = os.path.join(OUT, name)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
        size_mb = sum(f.stat().st_size for f in __import__('pathlib').Path(dst).rglob('*') if f.is_file()) / 1e6
        print(f"  {name:<20} {size_mb:.0f} MB")

gguf = "models/vulnreach_reranker_q4.gguf"
if os.path.exists(gguf):
    shutil.copy(gguf, os.path.join(OUT, "vulnreach_reranker_q4.gguf"))
    print(f"  GGUF: {round(os.path.getsize(gguf)/1e9, 2)} GB")

print("\nDownload from the Output tab →")

## Timing reference

| Scenario | DAPT | SFT | Total |
|---|---|---|---|
| Data pre-uploaded, 2×T4 | 5–7 hrs | 1.5–2.5 hrs | **~7–10 hrs** |
| Data downloaded live, 2×T4 | 5–7 hrs | 1.5–2.5 hrs | **~9–12 hrs** |
| Data pre-uploaded, A100 | 2–3 hrs | 45–75 min | **~3–5 hrs** |

Free Kaggle sessions last ~9–12 hrs. Pre-uploading data is the difference between
finishing in one session or needing two.